# EpistemicTrap-Metacog: Confidence Calibration Chains (CCC)

**License:** CC0 (Public Domain Dedication)

## 1. Goal & Faculty Isolation
**DeepMind Cognitive Faculty:** Section 7.7.2 (Metacognitive Monitoring - Calibration).

This benchmark isolate's a model's ability to monitor the accumulation of uncertainty across a long, multi-step reasoning chain. We test for **Epistemic Drift** - the systematic failure of a model to decrease its stated confidence as its "internal gears" encounter more probabilistic unknowns.

### 2. Methodology & Reproducibility (Competition Rule 2.8)
**Data Collection:** Multi-step probabilistic reasoning chains (N=100) with dependent outcomes.
**Audit Mechanism:** Proper scoring via Brier loss: compares stated confidence to step correctness, then maps to a [0,1] score.
**Hardware:** Standard Kaggle GPU/CPU kernel.

### Novel Insight
**What can we see now?** This task reveals whether a model's confidence is genuinely calculating probability or merely reflecting a fixed "Persona Confidence." We see if the model can acknowledge the *diminishing reliability* of its own logic over time.

---
**Kaggle Domain:** Clinical, Financial, and Forensic Probabilistic Reasoning  
**Primary Track:** Metacognition (Track 2)  
**Task Name:** EpistemicTrap-Metacognition-CCC

### Step 1: Scientific Environment Setup
We use the `kaggle_benchmarks` SDK for robust evaluation.

In [ ]:
# EpistemicTrap CCC - Setup
%pip install -q matplotlib seaborn
import os
os.environ['RENDER_SUBRUNS'] = 'False'
import kaggle_benchmarks as kbench
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
MODELS_TO_TEST = [kbench.llm]

### Step 2: High-Complexity Probabilistic Dataset
We use CCC items that require at least 3-5 dependent reasoning steps involving probability or scientific uncertainty.

In [ ]:
import glob
json_paths = glob.glob('/kaggle/input/**/*.json', recursive=True)
dataset_path = next((p for p in json_paths if 'metacog_dataset.json' in p), None)
if not dataset_path: raise FileNotFoundError('Could not find metacog_dataset.json in /kaggle/input/')
with open(dataset_path) as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df_ccc = df[df['subtype'] == 'CCC'].reset_index(drop=True)

print(f'Loaded CCC Calibration sets: {len(df_ccc)} chains.')

### Step 3: Calibration Fidelity Scoring (The Gradient)
A continuous score [0-1] based on calibration: average per-step **Brier loss** (confidence vs correctness), mapped to [0,1] for leaderboard compatibility.

In [ ]:
def _clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def _parse_first_json_objects(text: str) -> list[dict]:
    objs = []
    for m in re.finditer(r"\{[\s\S]*?\}", text):
        try:
            objs.append(json.loads(m.group(0)))
        except Exception:
            continue
    return objs

def _normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

def _as_float(s: str):
    try:
        return float(s)
    except Exception:
        return None

def _compare_answers(pred: str, gold: str) -> bool:
    if pred is None:
        return False
    p = _normalize_space(str(pred))
    g = _normalize_space(str(gold))

    m_g = re.fullmatch(r"(-?\d+)\s*/\s*(-?\d+)", g)
    if m_g:
        m_p = re.fullmatch(r"(-?\d+)\s*/\s*(-?\d+)", p)
        if not m_p:
            return False
        gn, gd = int(m_g.group(1)), int(m_g.group(2))
        pn, pd = int(m_p.group(1)), int(m_p.group(2))
        if gd == 0 or pd == 0:
            return False
        return gn * pd == pn * gd

    fg = _as_float(g)
    fp = _as_float(p)
    if fg is not None and fp is not None:
        return abs(fg - fp) <= 1e-3

    return p == g

@kbench.task(name='EpistemicTrap-Metacognition-CCC', version=2)
def metacog_ccc(llm, prompt: str, steps: list, final_answer: str, answer_type: str, **kwargs) -> float:
    try:
        response = llm.prompt(prompt)
        objs = _parse_first_json_objects(response)
        by_step = {}
        for o in objs:
            try:
                step = int(o.get('step'))
            except Exception:
                continue
            by_step[step] = o

        if not steps:
            return 0.0

        briers = []
        for gold_step in steps:
            i = int(gold_step.get('step'))
            gold = gold_step.get('answer')
            o = by_step.get(i, {})
            pred = o.get('answer')
            conf = o.get('confidence')
            try:
                p = _clamp01(float(conf) / 100.0)
            except Exception:
                p = 0.5
            y = 1.0 if _compare_answers(pred, gold) else 0.0
            briers.append((p - y) ** 2)

        if not briers:
            return 0.0
        avg_brier = sum(briers) / len(briers)
        return max(0.0, 1.0 - 4.0 * avg_brier)
    except Exception as e:
        print(f"Critical Evaluation Error: {e}")
        return 0.0

### Step 4: Full-Scale Execution
Executing the full dataset to ensure scientific significance.

In [ ]:
print(f"Evaluating EpistemicTrap-Metacognition-CCC [N={len(df_ccc)}]...")
with kbench.client.enable_cache():
    runs = metacog_ccc.evaluate(
        llm=MODELS_TO_TEST,
        evaluation_data=df_ccc,
    )

### Step 5: Insights & Chain Profiling
A smooth KDE distribution illustrates the "Gradient of Performance."

In [ ]:
try:
    df_res = runs.as_dataframe()
    df_res['score'] = df_res['result'].apply(lambda x: float(x) if x is not None else 0.0)
    
    plt.figure(figsize=(12, 6))
    # Visualizing Epistemic Drift mapping
    sns.kdeplot(data=df_res, x='score', fill=True, color='teal', cut=0, alpha=0.6)
    
    plt.title('Cognitive Profile: Chain Calibration Fidelity (CCC)', fontsize=16, pad=20)
    plt.xlabel('Fidelity Score (Average: {:.2f})'.format(df_res['score'].mean()), fontsize=12)
    plt.ylabel('Performance Density', fontsize=12)
    plt.xlim(-0.1, 1.1)
    plt.show()
except Exception as e:
    print('Analytics error:', e)

%choose EpistemicTrap-Metacognition-CCC